# Iris Flower Classification - Exploratory Data Analysis & Machine Learning

This notebook provides a complete data science workflow for the **Iris Flower Classification** task as part of the CodeAlpha Internship. 

### Objectives:
1. Load and explore the Iris dataset.
2. Perform detailed **Exploratory Data Analysis (EDA)** with beautiful, modern visualizations.
3. Preprocess and split the dataset.
4. Train and evaluate multiple machine learning classifiers (`Scikit-Learn`).
5. Select, analyze, and save the best-performing model.

## 1. Importing Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Set visualization style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12
print("Libraries imported successfully!")

## 2. Data Loading & Inspection

In [ ]:
# Load the Iris dataset
iris = load_iris()
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df['species'] = [iris.target_names[x] for x in iris.target]

# Display first 5 rows
df.head()

In [ ]:
# Check shape and summary of columns
print(f"Dataset Shape: {df.shape}")
print("\n--- Column Information ---")
df.info()

# Check for missing values
print("\n--- Missing Values ---")
print(df.isnull().sum())

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Statistical summary of the dataset
df.describe().T

In [ ]:
# Species distribution
print("Species Distribution:")
print(df['species'].value_counts())

### A. Visualizing Pairwise Relationships
A pairwise scatter plot (`pairplot`) is excellent for visualising how different features separate the three species of Iris flowers.

In [ ]:
sns.pairplot(df, hue="species", palette="husl", markers=["o", "s", "D"])
plt.suptitle("Pairwise Relationships of Iris Features", y=1.02)
plt.show()

### B. Correlation Heatmap
Let's visualize the correlation between the numeric features to see which variables are highly linear.

In [ ]:
plt.figure(figsize=(8, 6))
numeric_cols = df.drop(columns=['species'])
sns.heatmap(numeric_cols.corr(), annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Feature Correlation Matrix")
plt.show()

### C. Violin Plots & Box Plots
Violin and box plots help us understand the distribution density and outliers of each feature across different species.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
features = df.columns[:-1]

for i, feat in enumerate(features):
    row = i // 2
    col = i % 2
    sns.violinplot(ax=axes[row, col], x='species', y=feat, data=df, palette='muted')
    axes[row, col].set_title(f'{feat.capitalize()} by Species')

plt.tight_layout()
plt.show()

## 4. Data Preprocessing & Splitting

In [ ]:
# Split features and target
X = iris.data
y = iris.target

# Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

In [ ]:
# Feature Scaling using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data scaled successfully using StandardScaler!")

## 5. Machine Learning Models - Comparison

In [ ]:
# Define models
models = {
    "Logistic Regression": LogisticRegression(max_iter=200, random_state=42),
    "Support Vector Classifier": SVC(probability=True, random_state=42),
    "Random Forest Classifier": RandomForestClassifier(n_estimators=100, random_state=42),
    "k-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5)
}

# Train and compare
results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f"{name}: Accuracy = {acc:.4%}")

In [ ]:
# Visual Comparison
model_names = list(results.keys())
accuracies = list(results.values())

plt.figure(figsize=(8, 5))
colors = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2']
bars = plt.bar(model_names, accuracies, color=colors, width=0.5)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.01, f"{yval:.2%}", ha='center', va='bottom', fontweight='bold')

plt.ylabel("Accuracy")
plt.title("Machine Learning Model Accuracy Comparison", fontsize=14, fontweight='bold')
plt.ylim(0, 1.15)
plt.show()

## 6. Detailed Best Model Evaluation (SVC)

In [ ]:
# Evaluate SVC (the best model)
best_model = SVC(probability=True, random_state=42)
best_model.fit(X_train_scaled, y_train)
y_pred = best_model.predict(X_test_scaled)

# Accuracy & Classification Report
print(f"SVC Test Accuracy: {accuracy_score(y_test, y_pred):.2%}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=iris.target_names, yticklabels=iris.target_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix for Best Model (SVC)", fontsize=14, fontweight='bold')
plt.show()

## Conclusion

We have successfully built a high-performing classification model for the **Iris Flower Classification** task. 

- **Exploratory Data Analysis** showed that species *Iris Setosa* is linearly separable from the other two species using Petal features, whereas *Iris Versicolor* and *Iris Virginica* have slight overlap.
- **Support Vector Classifier (SVC)** achieved the best test accuracy of **96.67%**, making it our production model of choice.
- The serialized model and preprocessor scaling weights have been saved to the `models/` directory for robust deployment in the **Streamlit Web Application**.